# Multi30k EN BPE lengths

Prints BPE length stats (mean, min, max, percentiles) for each Multi30k EN split.

Lengths are counted from the datasets `get_multi30ken_mlm_dataloader` builds, so they
use the same tokenizer as training: the default one if set, otherwise a freshly
trained one. The datasets are padded to `MAX_SEQ_LEN`, and a length is the number of
non-`PAD_ID` ids in a row. Rows longer than `MAX_SEQ_LEN` would be truncated, so the
notebook refuses to report stats if any row reaches it.

Run it from the repo root as the [hello_world notebook](hello_world.ipynb) describes:
the first cell puts `app/src` on the path.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "app" / "src").is_dir()
)
sys.path.insert(0, str(repo_root / "app" / "src"))  # noqa: NAR001

In [ ]:
import numpy as np
from loguru import logger
from training.constants import TEST_SEED, VAL_SEED
from training.dataload.datasets import InMemoryDataset
from training.dataload.multi30k import get_multi30ken_mlm_dataloader
from training.preprocess.constants import PAD_ID

MAX_SEQ_LEN = 512
PERCENTILES = (1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9)

## Count non-pad ids per row

A row that fills all `MAX_SEQ_LEN` positions may have been truncated, so its length
would be undercounted. `bpe_lengths` raises instead of reporting such a split.

In [ ]:
def bpe_lengths(dataset: InMemoryDataset) -> np.ndarray:
    lengths = np.array(
        object=[int((dataset[i][1] != PAD_ID).sum()) for i in range(len(dataset))]  # noqa: NAR001
    )
    if lengths.max() >= MAX_SEQ_LEN:
        raise ValueError(  # noqa: NAR001
            f"a row has {lengths.max()} ids, reaching MAX_SEQ_LEN={MAX_SEQ_LEN}; "
            "raise MAX_SEQ_LEN so no row is truncated"
        )
    return lengths


def log_stats(split: str, lengths: np.ndarray) -> None:
    logger.info(  # noqa: NAR001
        "{:<5} n={:>6}  mean={:6.2f}  min={:>3}  max={:>3}",
        split,
        len(lengths),  # noqa: NAR001
        lengths.mean(),
        lengths.min(),
        lengths.max(),
    )
    values = np.percentile(a=lengths, q=PERCENTILES)
    logger.info(  # noqa: NAR001
        "{:<5} {}",
        split,
        "  ".join(f"p{q:g}={v:.1f}" for q, v in zip(PERCENTILES, values)),  # noqa: NAR001
    )

## Load every row of each split and log its stats

`train_num=None`, `val_num=None` and `test_num=None` keep every row, and
`batch_size=1` is irrelevant here because only each loader's dataset is read.

In [ ]:
train_loader, val_loader, test_loader, tokenizer = get_multi30ken_mlm_dataloader(
    seq_len=MAX_SEQ_LEN,
    val_seed=VAL_SEED,
    test_seed=TEST_SEED,
    batch_size=1,
    train_num=None,
    val_num=None,
    test_num=None,
)
logger.info("Tokenizer has {} merges", len(tokenizer.merges))  # noqa: NAR001
for split, loader in (("train", train_loader), ("val", val_loader), ("test", test_loader)):
    dataset = loader.dataset
    assert isinstance(dataset, InMemoryDataset)  # noqa: NAR001
    log_stats(split=split, lengths=bpe_lengths(dataset=dataset))